# 02. 手写一个 Agent Loop

AGDR VLA 入门教程 · 第 3 章 · 前半周

- **学习目标**：理解 Agent 的核心是一个「调用模型 → 执行工具 → 回传结果」的循环，并亲手用不到 100 行 Python 实现它
- **前置知识**：[01-大模型API与消息格式](./01-大模型API与消息格式.ipynb)，特别是 OpenAI 格式的工具调用
- **预计学习时长**：2 小时

## Agent 就是一个循环

上一节我们**手动**完成了一次工具调用：发请求 → 看到 `tool_calls` → 执行工具 → 回传结果 → 再发请求。

如果模型拿到结果后，又想调用另一个工具呢？那就再来一遍。把这件事交给程序自动重复，就得到了 Agent：

```text
messages = [系统提示词, 用户任务]
循环:
    reply = 调用模型(messages)
    把 reply 追加到 messages
    如果 reply 里没有工具调用:
        任务完成，返回 reply 的文字      ← 模型自己决定什么时候停
    否则，对每一个工具调用:
        执行工具，把结果追加到 messages
```

Claude Code、Codex 这类工具，最核心的部分就是这个循环。它们多出来的是更多的工具、权限确认、上下文压缩等，这些在后面几节逐一讲。

本节用 **OpenAI 格式**实现，因为兼容它的服务最多。动手试试里有一道题，是把它改成 Anthropic 格式。

## 准备工作

第一个单元格和 01 节相同：读取环境变量，定义发送请求的函数。

新增的 `call_model` 多了一个离线机制：离线模式下，它按顺序从 `offline_script` 里取出预先录制的响应。这样即使没有 key，循环也能完整跑起来，**工具也会真的执行**，只有模型的决定是录制好的。

In [ ]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "")
ONLINE = bool(OPENAI_API_KEY and OPENAI_MODEL)
SAMPLES = Path("data/samples").resolve()

print("模式:", "在线" if ONLINE else "离线（读取录制的模型响应）")


def show(obj):
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def post_json(url, headers, body):
    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json", **headers},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", "replace")
        raise RuntimeError(f"HTTP {error.code}: {detail}") from None


def call_model(messages, tools, offline_script):
    """调用一次模型。离线模式下，从 offline_script 里按顺序取出录制的响应。"""
    if not ONLINE:
        if not offline_script:
            raise RuntimeError("离线脚本已用完：离线模式只能运行本节预设的任务")
        sample = offline_script.pop(0)
        return json.loads((SAMPLES / sample).read_text(encoding="utf-8"))
    url = OPENAI_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"}
    body = {"model": OPENAI_MODEL, "messages": messages, "tools": tools}
    return post_json(url, headers, body)

### 准备一个沙箱工作区

Agent 要读文件、运行命令，我们不让它碰真实的项目，而是在系统临时目录里建一个小工作区，放一份模拟的训练日志。

In [ ]:
import shutil
import tempfile

WORKSPACE = (Path(tempfile.gettempdir()) / "agent-demo").resolve()
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir()

(WORKSPACE / "README.md").write_text("# 机械臂抓取实验\n\n训练日志见 train.log，超参数见 config.json。\n", encoding="utf-8")
(WORKSPACE / "config.json").write_text('{"lr": 0.0003, "batch_size": 32, "epochs": 5}\n', encoding="utf-8")
(WORKSPACE / "train.log").write_text(
    """[INFO] epoch=1 loss=2.31
[WARN] gpu_memory_high
[INFO] epoch=2 loss=1.87
[ERROR] dataloader_timeout
[INFO] epoch=3 loss=1.42
[ERROR] camera_frame_dropped
[INFO] epoch=4 loss=0.98
[ERROR] dataloader_timeout
[INFO] epoch=5 loss=1.05
""",
    encoding="utf-8",
)
print("工作区:", WORKSPACE)
print(sorted(p.name for p in WORKSPACE.iterdir()))

## 第一步：定义工具

一个工具由两部分组成：

1. **Python 函数**：真正干活的代码
2. **工具说明书**（JSON Schema）：发给模型，告诉它工具叫什么、做什么、要什么参数

我们先做两个只读的工具：`list_dir` 列出目录，`read_file` 读取文件。

注意 `safe_path`：模型给出的路径**不可信**，它可能写出 `../../etc/passwd` 这样的路径。工具必须自己检查边界，只允许访问工作区里的文件。

In [ ]:
def safe_path(path):
    """把模型给的相对路径转成工作区内的绝对路径；越界就报错"""
    full = (WORKSPACE / path).resolve()
    if full != WORKSPACE and WORKSPACE not in full.parents:
        raise ValueError(f"不允许访问工作区之外的路径: {path}")
    return full


def list_dir(path="."):
    target = safe_path(path)
    names = sorted(p.name + ("/" if p.is_dir() else "") for p in target.iterdir())
    return "\n".join(names) or "(空目录)"


def read_file(path):
    return safe_path(path).read_text(encoding="utf-8")


print(list_dir("."))
print("---")
print(read_file("config.json"))

In [ ]:
def tool_spec(name, description, properties, required):
    """生成一份 OpenAI 格式的工具说明书，省得每次都写一长串"""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }


TOOL_SPECS = [
    tool_spec("list_dir", "列出工作区中某个目录下的文件，目录名以 / 结尾",
              {"path": {"type": "string", "description": "相对于工作区的路径，根目录是 ."}}, ["path"]),
    tool_spec("read_file", "读取工作区中一个文本文件的全部内容",
              {"path": {"type": "string", "description": "相对于工作区的文件路径"}}, ["path"]),
]

# 工具名 → 函数。模型说要调用哪个名字，就从这里找到对应的函数
TOOLS = {"list_dir": list_dir, "read_file": read_file}

### 执行工具：出错也要告诉模型

`execute_tool` 负责「按名字找到函数并调用」。这里有一个重要的设计：**工具出错时不让程序崩溃，而是把错误信息当作结果返回给模型。**

模型看到「文件不存在」之后，往往会自己换个路径再试一次。这种「看到错误 → 自我修正」的能力，正是 Agent 比一问一答强的地方。

In [ ]:
def execute_tool(name, arguments_json):
    try:
        args = json.loads(arguments_json)  # OpenAI 格式的参数是 JSON 字符串
        if name not in TOOLS:
            return f"错误：没有名为 {name} 的工具"
        return str(TOOLS[name](**args))
    except Exception as error:  # 任何异常都转成文字交给模型
        return f"错误：{type(error).__name__}: {error}"


print(execute_tool("read_file", '{"path": "train.log"}')[:60], "...")
print(execute_tool("read_file", '{"path": "no_such.txt"}'))
print(execute_tool("read_file", '{"path": "../../etc/passwd"}'))
print(execute_tool("fly", "{}"))

## 第二步：写出循环

下面就是本节的主角。对照开头的伪代码读一遍，每一行都能找到对应。

几个细节：

- `max_steps`：最多调用多少次模型。没有它，一个出错的 Agent 可能无限循环，一直消耗额度
- `finish_reason == "length"`：回答被截断了，不能当作正常完成
- `print` 出来的每一步，就是你在 Claude Code 里看到的那些「正在读取文件……」的提示

In [ ]:
SYSTEM_PROMPT = """你是一个帮助分析机器人训练实验的助手。
你可以使用工具查看工作区中的文件。先用工具收集信息，再回答；不要猜测文件内容。
回答使用中文，简洁明了。"""


def run_agent(task, offline_script=None, max_steps=10):
    offline_script = list(offline_script or [])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    for step in range(1, max_steps + 1):
        reply = call_model(messages, TOOL_SPECS, offline_script)
        choice = reply["choices"][0]
        message = choice["message"]
        messages.append(message)  # 模型的回复（包括工具调用请求）必须放回历史

        print(f"── 第 {step} 步 · 输入 {reply['usage']['prompt_tokens']} token")
        if message.get("content"):
            print("模型:", message["content"])

        tool_calls = message.get("tool_calls") or []
        if not tool_calls:  # 没有工具调用 → 模型认为任务完成
            if choice["finish_reason"] == "length":
                print("⚠️ 回答被截断了")
            return message.get("content"), messages

        for call in tool_calls:
            name = call["function"]["name"]
            arguments = call["function"]["arguments"]
            print(f"调用工具: {name}({arguments})")
            result = execute_tool(name, arguments)
            preview = result if len(result) <= 120 else result[:120] + "……"
            print("工具结果:", preview.replace("\n", " | "))
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})

    print(f"⚠️ 达到最大步数 {max_steps}，强制停止")
    return None, messages

### 运行

给 Agent 一个任务。注意：**我们没有告诉它日志文件叫什么**，它需要自己先看看工作区里有什么。

In [ ]:
answer, history = run_agent(
    "工作区里的训练日志中，最低的 loss 是多少？出现在第几个 epoch？",
    offline_script=["agent_list_dir.json", "agent_read_log.json", "agent_loss_answer.json"],
)
print("\n最终回答:", answer)

Agent 自己完成了「先列目录 → 发现 train.log → 读取 → 回答」三步。打开 `train.log` 核对一下，答案是否正确？

> 模型的回答不一定对。即使它读到了正确的文件，也可能算错、看错行。**用 Agent 得到的结论，要养成核对的习惯**，第 08 节会专门讲。

### 看看消息历史长成了什么样

这就是模型在最后一步「看到」的全部内容。每调用一次工具，历史就多两条：一条 `assistant`（调用请求），一条 `tool`（结果）。

In [ ]:
for i, m in enumerate(history):
    if m.get("tool_calls"):
        text = "调用 " + ", ".join(c["function"]["name"] for c in m["tool_calls"])
    else:
        text = (m.get("content") or "").replace("\n", " | ")
    print(f"{i}. {m['role']:9s} {text[:70]}")
print("\n共", len(history), "条消息")

回看运行时打印的「输入 token」：每一步都比上一步多，因为整段历史每次都要重新发送。这里只读了一个很小的文件；如果 Agent 读了一个几千行的源代码文件，这些内容会一直留在历史里，**之后的每一步都要为它付费**。这是第 03 节要解决的问题。

## 第三步：加一个「危险」的工具

只能读文件的 Agent 用处有限。真实的编程 Agent 都能**运行命令**，这也是它们强大和危险的根源：模型可以写出任何命令，包括 `rm -rf`。

所以我们给 `run_command` 加上两道保护：

1. **执行前让人确认**：打印出模型想运行的命令，由你输入 `y` 才执行。拒绝时，把「用户拒绝」作为结果告诉模型
2. **限制输出长度和运行时间**：输出太长会撑爆上下文，命令卡住会卡死循环

这就是 Claude Code、Codex 运行命令前向你请求批准的原理。

In [ ]:
import subprocess

MAX_OUTPUT = 2000  # 最多返回给模型的字符数


def run_command(command):
    print(f"\n🤖 Agent 想在 {WORKSPACE} 中运行：\n    {command}")
    if input("允许执行吗？输入 y 执行，其他任意键拒绝：").strip().lower() != "y":
        return "用户拒绝执行这条命令。"
    completed = subprocess.run(
        command, shell=True, cwd=WORKSPACE,
        capture_output=True, text=True, timeout=30,
    )
    output = completed.stdout + completed.stderr
    if len(output) > MAX_OUTPUT:
        output = output[:MAX_OUTPUT] + f"\n……（输出过长，已截断，共 {len(output)} 字符）"
    return f"退出码 {completed.returncode}\n{output}"


TOOL_SPECS.append(tool_spec(
    "run_command", "在工作区目录中运行一条 shell 命令，返回退出码和输出",
    {"command": {"type": "string", "description": "要运行的命令"}}, ["command"],
))
TOOLS["run_command"] = run_command
print("当前工具:", list(TOOLS))

运行下面的单元格，会弹出输入框。**先读懂命令再决定**，这和你以后使用真实 Agent 时完全一样。

- 第一次运行时输入 `y`，看看正常流程
- 再运行一次，这次直接回车拒绝，观察工具结果变成了什么

> Windows 用户：`shell=True` 在 Windows 上使用的是 cmd，没有 `grep`，命令会执行失败。这时候注意看：错误信息被当作结果交给了模型。在线模式下，模型往往会换一条命令再试。

> 离线模式下，模型的最终回答是录制好的，不会因为你拒绝而改变；在线模式下，模型会根据「用户拒绝」调整回答。

In [ ]:
answer, history = run_agent(
    "用命令统计 train.log 里有多少行 ERROR。",
    offline_script=["agent_run_grep.json", "agent_grep_answer.json"],
)
print("\n最终回答:", answer)

## 循环什么时候停

| 停止的原因 | 在代码里的体现 | 怎么处理 |
| --- | --- | --- |
| 模型认为任务完成 | 回复里没有 `tool_calls` | 正常返回 |
| 达到最大步数 | `for` 循环结束 | 强制停止，提示用户 |
| 回答被截断 | `finish_reason == "length"` | 不能当作完成，需要提示或重试 |
| 请求出错 | `post_json` 抛出异常 | 网络问题可以重试；key 错误、额度不足要人来处理 |

要特别注意第一条：**什么时候停，是模型自己决定的。** 你只能通过系统提示词引导它，通过 `max_steps` 兜底。

## 和真实的 Agent 差在哪

我们的 Agent 不到 100 行，但已经具备了核心结构。真实的编程 Agent 在它的基础上，还多了这些：

| 能力 | 解决什么问题 | 本章哪一节讲 |
| --- | --- | --- |
| 更多、更好用的工具：编辑文件的部分内容、搜索代码、联网 | 让 Agent 能真正改代码 | — |
| 细粒度的权限系统：哪些工具自动允许，哪些每次都问 | 安全与效率的平衡 | 09 |
| 上下文压缩 | 历史太长、太贵 | 03 |
| MCP | 用统一的方式接入外部工具 | 04 |
| Skill、Hook | 按需加载专业知识；在固定时机自动执行检查 | 05 |
| 项目说明文件（`AGENTS.md` 等） | 每次都要交代的项目背景，自动加进系统提示词 | 07 |
| 流式输出、并行调用工具、子 Agent | 更快、能处理更大的任务 | — |

所以下次你看到 Claude Code 在屏幕上「读取文件 → 运行测试 → 修改代码 → 再运行测试」时，你知道它背后在做什么了。

## 动手试试

不计分。前四题在离线模式下也能完成。

1. 把第一个任务的 `max_steps` 改成 `1` 再运行，会发生什么？为什么？
2. 不运行代码，根据本节的流程回答：第一个任务一共调用了几次模型 API？最终的 `history` 有几条消息？然后运行「看看消息历史」那个单元格核对。
3. `run_command` 里，如果把 `timeout=30` 去掉，模型又让 Agent 运行了 `tail -f train.log`，会发生什么？（提示：第 2 章第 03 节）
4. 为什么 `read_file` 里需要 `safe_path`，而 `run_command` 里的 `cwd=WORKSPACE` **不能**起到同样的保护作用？
5. （在线，进阶）把循环改成 Anthropic 格式。先列出要改的地方，再动手。

<details>
<summary>参考答案</summary>

**第 1 题**：第 1 步模型请求调用 `list_dir`，工具执行了，但随后循环就结束了，输出「达到最大步数 1，强制停止」，`answer` 是 `None`。模型还没来得及看到工具结果，更没有机会回答。`max_steps` 太小会让 Agent 半途而废，太大则可能浪费额度，要按任务复杂度设置。

**第 2 题**：调用了 3 次模型（列目录、读文件、回答）。消息有 7 条：`system`、`user`，再加 2 组「`assistant` 调用 + `tool` 结果」共 4 条，最后 1 条 `assistant` 回答。

**第 3 题**：`tail -f` 会一直等待新内容，永远不会退出。没有超时，`subprocess.run` 就会一直阻塞，整个 Agent 卡死，只能手动中断 notebook。这就是要设置超时的原因。

**第 4 题**：`cwd` 只决定命令**从哪个目录开始**运行，命令本身仍然可以写 `cat ../../某个文件` 或绝对路径，访问任何位置。shell 命令几乎无法靠检查字符串来限制，所以只能依靠**人工确认**，或者把整个 Agent 放进隔离环境（容器、虚拟机）里运行。

**第 5 题**：需要改这些地方：

- 地址、请求头改成 Anthropic 格式（参考 01 节的 `call_anthropic`）
- 系统提示词从 `messages` 移到顶层的 `system` 字段，并加上必填的 `max_tokens`
- 工具说明书改成 `{name, description, input_schema}`
- 判断是否结束：看 `stop_reason` 是否为 `tool_use`，而不是看 `tool_calls`
- 从 `content` 里找出所有 `tool_use` 块，参数 `input` 已经是字典，不用 `json.loads`
- 所有工具结果包装成 `tool_result` 块，放进**同一条** `role: user` 的消息里

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 用伪代码写出 Agent Loop，并说出每一步的作用 | Agent 就是一个循环 |
| 为一个 Python 函数写出工具说明书，并注册到 Agent 中 | 第一步：定义工具 |
| 解释为什么工具出错时要把错误交给模型，而不是让程序崩溃 | 执行工具 |
| 说出循环的几种停止条件，以及 `max_steps` 的作用 | 循环什么时候停 |
| 解释为什么运行命令前需要人工确认，以及输出截断、超时的意义 | 第三步 |
| 说出消息历史为什么越来越长 | 看看消息历史 |

下一节：[03-上下文管理与压缩](./03-上下文管理与压缩.ipynb)